# Evaluate TCIA Dataset

In [ ]:
import os, torch

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["KERAS_BACKEND"] = "torch"

# initialize dataset parameters
data_dir = os.path.normpath('/home/qhe2/Documents/Datasets/TCIA/processed/test')

# initialize model parameters
device = torch.device('cuda')
model_path = os.path.normpath('experiments/tcia_seg.exp/checkpoints/last.keras')

## Load Dataset

In [ ]:
from data import ProcessedTCIA

dataset = ProcessedTCIA(data_dir)

In [ ]:
x, y = dataset[6]
x = x.unsqueeze(0)
y = y.unsqueeze(0)

In [ ]:
x.shape, y.shape

In [ ]:
x.max(), x.min(), y.max(), y.min()

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

x_np = x[0].detach().cpu().numpy()
x_np = (x_np - x_np.min()) / (x_np.max() - x_np.min())
y_np = y[0].detach().cpu().numpy()

# prepare label display
is_scalar_label = np.ndim(y_np) == 0 or (y_np.ndim == 1 and y_np.size == 1)

if is_scalar_label:
    plt.figure(figsize=(6,6))
    plt.imshow(x_np, cmap='gray' if x_np.ndim == 2 else None)
    plt.title(f"label: {int(np.squeeze(y_np))}")
    plt.axis('off')
else:
    plt.figure(figsize=(10,5))
    plt.subplot(1,2,1)
    plt.imshow(x_np, cmap='gray' if x_np.ndim == 2 else None)
    plt.title('image')
    plt.axis('off')

    plt.subplot(1,2,2)
    # convert label channels if needed
    if y_np.ndim == 3 and y_np.shape[0] in (1,3):
        lbl = y_np.squeeze(0) if y_np.shape[0] == 1 else np.transpose(y_np, (1,2,0))
    else:
        lbl = y_np
    plt.imshow(lbl)
    plt.title('label')
    plt.axis('off')

plt.show()

## Load Model

In [ ]:
import keras

print("keras backend:", keras.backend.backend())

In [ ]:
from comp2comp.finetune.metrics import SparseMeanDice
from comp2comp.finetune.optim import LayerwiseAdam

In [ ]:
import keras
from typing import cast

model = cast(keras.Model, keras.models.load_model(model_path))

## Predict

In [ ]:
pred: torch.Tensor = model(x)

In [ ]:
pred = pred.argmax(dim=-1)

In [ ]:
pred.shape

In [ ]:
pred_np = pred[0].cpu().detach().numpy()

plt.figure(figsize=(6,6))
# convert label channels if needed
if pred_np.ndim == 3 and pred_np.shape[0] in (1,3):
    lbl = pred_np.squeeze(0) if pred_np.shape[0] == 1 else np.transpose(pred_np, (1,2,0))
else:
    lbl = pred_np
plt.imshow(lbl)
plt.title('label')
plt.axis('off')

In [ ]:
from torch.utils.data import DataLoader, Sampler

data_loader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=False,
    num_workers=min(4, os.cpu_count() or 1),
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2,
)

In [ ]:
model.evaluate(data_loader, verbose=1, return_dict=True)